In [1]:
import numpy as np
import matplotlib.pyplot as plt
from cellpose import models
from cellpose.io import imread
from pathlib import Path
from livecell_tracker.preprocess.utils import normalize_img_to_uint8
from napari.layers import Shapes
#from livecell_tracker.segment.cellpose_utils import segment_single_images_by_cellpose, segment_single_image_by_cellpose

In [2]:
from livecell_tracker.core.datasets import LiveCellImageDataset, SingleImageDataset


# change path to data path
dataset_dir_path = Path(r"d:\Xinglab\data\tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19\XY1")
mask_dataset_path = Path(r"d:\Xinglab\data\tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19\out\XY1\seg")


import glob
time2url = sorted(glob.glob(str((Path(dataset_dir_path) / Path("*_DIC.tif")))))
time2url = {i: path for i, path in enumerate(time2url)}
dic_dataset = LiveCellImageDataset(time2url=time2url, ext="tif")
#dic_dataset = LiveCellImageDataset(dataset_dir_path, ext="tif")
#dic_dataset.time2url
mask_dataset = LiveCellImageDataset(mask_dataset_path, ext="png")
#mask_dataset.time2url


302 png img file paths loaded;


In [3]:
from livecell_tracker.core import SingleCellTrajectory, SingleCellStatic
from livecell_tracker.segment.ou_utils import create_ou_input_from_sc
from livecell_tracker.segment.utils import find_contours_opencv
from livecell_tracker.core.datasets import SingleImageDataset

class ScSegOperator:
    """
    A class for performing segmentation on single cell images.

    Attributes
    ----------
    viewer : napari.Viewer
        The napari viewer.
    single_cell_static : SingleCellStatic
        The single cell static object.
    shape_layer : napari.layers.Shapes
        The napari shape layer for displaying the segmentation.
    """

    MANUAL_CORRECT_SEG_MODE = 0
    CSN_CORRECT_SEG_MODE = 1
    
    def __init__(
        self,
        sc: SingleCellStatic,
        viewer,
        shape_layer: Shapes=None,
        face_color=(0, 0, 1, 1),
        magicgui_container = None,
        csn_model = None,
    ):
        """
        Parameters
        ----------
        viewer : napari.Viewer
            The napari viewer.
        single_cell_static : SingleCellStatic
            The single cell static object.
        """
        
        self.sc = sc
        self.viewer = viewer
        self.shape_layer = shape_layer
        self.face_color = face_color
        self.mode = self.MANUAL_CORRECT_SEG_MODE
        self.magicgui_container = magicgui_container
        self.csn_model = csn_model
        
        if not (self.shape_layer is None):
            self.setup_edit_contour_shape_layer()

    def create_sc_layer(self, name=None, contour_sample_num=100):
        if name is None:
            name = f"sc_{self.sc.id}"
        shape_vec = self.sc.get_napari_shape_contour_vec(contour_sample_num=contour_sample_num)
        properties = {"sc": [self.sc]}
        print("shape vec", shape_vec)
        shape_layer = self.viewer.add_shapes(
            [shape_vec],
            properties=properties,
            face_color=[self.face_color],
            shape_type="polygon",
            name=name,
        )
        self.shape_layer = shape_layer
        self.setup_edit_contour_shape_layer()
        print(">>> create sc layer done")

    def remove_sc_layer(self):
        if self.shape_layer is None:
            return
        self.viewer.layers.remove(self.shape_layer)
        self.shape_layer = None

    def update_shape_layer_by_sc(self, contour_sample_num=100):
        shape_vec = self.sc.get_napari_shape_contour_vec(contour_sample_num=contour_sample_num)
        self.shape_layer.data = [shape_vec]

    def correct_segment(self, model, create_ou_input_kwargs = {
            "padding_pixels": 50,
            "dtype": float,
            "remove_bg": False,
            "one_object": True,
            "scale": 0,
        }):
        import torch
        from torchvision import transforms
        #  padding_pixels=padding_pixels, dtype=dtype, remove_bg=remove_bg, one_object=one_object, scale=scale

        input_transforms = transforms.Compose(
                [
                    transforms.Resize(size=(412, 412)),
                ]
        )
        temp_sc = self.sc.copy()
        new_contour = np.array(self.shape_layer.data[0])
        new_contour = new_contour[:, -2:] # remove slice index (time)
        temp_sc.update_contour(new_contour)
        temp_sc.update_bbox()
        res_bbox = temp_sc.bbox
        ou_input = create_ou_input_from_sc(temp_sc, **create_ou_input_kwargs)
        # ou_input = create_ou_input_from_sc(self.sc, **create_ou_input_kwargs)
        original_shape = ou_input.shape

        ou_input = input_transforms(torch.tensor([ou_input]))
        ou_input = torch.stack([ou_input, ou_input, ou_input], dim=1)
        ou_input = ou_input.float().cuda()

        back_transforms = transforms.Compose(
            [
                transforms.Resize(size=(original_shape[0], original_shape[1])),
            ]
        )
        output = model(ou_input)
        output = back_transforms(output)
        return output, res_bbox

    def replace_sc_contour(self, contour, padding_pixels=0, refresh=True):
        self.sc.contour = contour + self.sc.bbox[:2] - padding_pixels
        self.sc.update_bbox()
        if refresh:
            self.update_shape_layer_by_sc()

    def setup_edit_contour_shape_layer(self):
        return
        # TODO 
        from copy import deepcopy
        # Callback to check if shape_layer has more than one shape and remove the last one
        self.saved_data = deepcopy(self.shape_layer.data)
        def _shape_data_changed(event):
            print("_shape_data_changed fired")
            print("len of shape_layer.data:", len(self.shape_layer.data))
            if len(self.shape_layer.data) > 1:
                # self.shape_layer.events.data.disconnect(self._shape_data_changed)  # disconnect the callback
                print("[_shape_data_changed] len of saved_data:", len(self.saved_data))
                self.shape_layer.data = deepcopy(self.saved_data)
                # self.shape_layer.events.data.connect(self._shape_data_changed)
            elif len(self.shape_layer.data) == 1:
                self.saved_data = deepcopy(self.shape_layer.data)
        # If the shape_layer already exists, connect the callback
        if self.shape_layer is not None:
            self.shape_layer.events.data.connect(_shape_data_changed)

    def show_selected_mode_widget(self):
        self.magicgui_container[0].show()
        self.magicgui_container[1].show()
        if self.mode == self.MANUAL_CORRECT_SEG_MODE:
            self.magicgui_container[2].show()
            self.magicgui_container[3].show()
            self.magicgui_container[4].show()
            self.magicgui_container[5].show()
            
    def hide_function_widgets(self):
        for i in range(2, len(self.magicgui_container)):
            self.magicgui_container[i].hide()

    def save_seg_callback(self):
        """Save the segmentation to the single cell object."""
        def _get_contour_from_shape_layer(layer: Shapes):
            """Get contour coordinates from a shape layer in napari."""
            vertices = np.array(layer.data[0])
            print("vertices:", vertices)

            # ignore the first vertex, which is the slice index
            vertices = vertices[:, 1:3]
            return vertices
        # Get the contour coordinates from the shape layer
        contour = _get_contour_from_shape_layer(self.shape_layer)
        # Store the contour in the single cell object
        self.sc.contour = contour
        self.sc.update_bbox()

    def csn_correct_seg_callback(self, padding_pixels = 50):
        print("csn_correct_seg_callback fired")
        create_ou_input_kwargs = {
            "padding_pixels": padding_pixels,
            "dtype": float,
            "remove_bg": False,
            "one_object": True,
            "scale": 0,
        }
        output, res_bbox = self.correct_segment(self.csn_model, create_ou_input_kwargs=create_ou_input_kwargs)
        bin_mask = output[0].cpu().detach().numpy()[0] > 0.5
        contours = find_contours_opencv(bin_mask.astype(bool))
        # contour = [0]
        new_shape_data = []
        for contour in contours:
            contour_in_original_image = contour + res_bbox[:2] - padding_pixels
            # replace the current shape_layer's data with the new contour
            napari_vertices = [[self.sc.timeframe] + list(point) for point in contour_in_original_image]
            napari_vertices = np.array(napari_vertices)
            new_shape_data.append((napari_vertices, "polygon"))

        self.shape_layer.data = []
        self.shape_layer.add(new_shape_data, shape_type=['polygon'])
        print("csn_correct_seg_callback done!")
    
    def clear_sc_layer_callback(self):
        self.shape_layer.data = []
        print("clear_sc_layer_callback done!")
    
    def restore_sc_contour_callback(self):
        self.update_shape_layer_by_sc()
        print("restore_sc_contour_callback done!")

In [4]:
from skimage.measure import regionprops
from livecell_tracker.segment.utils import prep_scs_from_mask_dataset
single_cells = prep_scs_from_mask_dataset(mask_dataset, dic_dataset)

  0%|          | 0/302 [00:00<?, ?it/s]

100%|██████████| 302/302 [01:06<00:00,  4.55it/s]


In [5]:
single_cells_by_time = {}
for cell in single_cells:
    if cell.timeframe not in single_cells_by_time:
        single_cells_by_time[cell.timeframe] = []
    single_cells_by_time[cell.timeframe].append(cell)

In [6]:
from typing import List
from livecell_tracker.track.sort_tracker_utils import (
    gen_SORT_detections_input_from_contours,
    update_traj_collection_by_SORT_tracker_detection,
    track_SORT_bbox_from_contours,
    track_SORT_bbox_from_scs
)
max_age = 5
min_hits = 3
traj_collection = track_SORT_bbox_from_scs(single_cells, dic_dataset, mask_dataset=mask_dataset, max_age=max_age, min_hits=min_hits)


matching image path: d:/Xinglab/data/tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19/XY1/CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19_T001_XY01_DIC.tif
matching image path: d:/Xinglab/data/tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19/XY1/CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19_T002_XY01_DIC.tif
matching image path: d:/Xinglab/data/tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19/XY1/CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19_T003_XY01_DIC.tif
matching image path: d:/Xinglab/data/tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19/XY1/CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19_T004_XY01_DIC.tif
matching image path: d:/Xinglab/data/tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19/XY1/CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19_T005_XY01_DIC.tif
matching image path: d:/Xinglab/data/tifs_CFP_A549-VIM_lessThan24hr_NoTreat_NA_YL_Ti2e_2022-10-19/XY1/CFP_A549-VIM_lessT

In [7]:
from livecell_tracker.model_zoo.segmentation.sc_correction import CorrectSegNet
ckpt = r"D:\Xinglab\LiveCellTracker-dev\notebooks\notebook_results\csn_models\model_v11_epoch=3282-test_loss=2.3688.ckpt"


model = CorrectSegNet.load_from_checkpoint(ckpt)
model = model.cuda()
model = model.eval()

Lightning automatically upgraded your loaded checkpoint from v1.8.6 to v2.0.3. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint --file D:\Xinglab\LiveCellTracker-dev\notebooks\notebook_results\csn_models\model_v11_epoch=3282-test_loss=2.3688.ckpt`
c:\Users\Gaohan\anaconda3\envs\checkcuda\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\Gaohan\anaconda3\envs\checkcuda\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DeepLabV3_ResNet50_Weights.COCO_WITH_VOC_LABELS_V1`. You can also use `weights=DeepLabV3_ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


>>> Using MSE loss
>>> Based on loss type, training output threshold:  1


In [8]:
%gui qt
from livecell_tracker.core.napari_visualizer import NapariVisualizer
import napari
from skimage import data
from livecell_tracker.core.single_cell import SingleCellStatic, SingleCellTrajectory, SingleCellTrajectoryCollection
import numpy as np
from napari.viewer import Viewer
from livecell_tracker.core.visualizer import Visualizer



# viewer = napari.view_image(dic_dataset.to_dask(), name='dic_image', cache=True)
# shape_layer = NapariVisualizer.viz_trajectories(traj_collection, viewer, contour_sample_num=20)

In [9]:
import napari
from livecell_tracker.core.sct_operator import SctOperator, create_sct_napari_ui
viewer = napari.view_image(dic_dataset.to_dask(), name="dic_image", cache=True)
shape_layer = NapariVisualizer.gen_trajectories_shapes(traj_collection, viewer, contour_sample_num=20)
shape_layer.mode = "select"

In [10]:

sct_operator = SctOperator(traj_collection, shape_layer, viewer)
# sct_operator.setup_shape_layer(shape_layer)
create_sct_napari_ui(sct_operator)

current shape layer shape properties:  Event
setting face color of selected shape...
<selection complete>
current shape layer shape properties:  Event
shape already selected, please select another shape
edit sc fired!
|-----? More than one shape is selected. The first selected shape is used for editing.
shape vec [[126, 630, 1427], [126, 632, 1423], [126, 634, 1419], [126, 636, 1415], [126, 640, 1412], [126, 644, 1410], [126, 648, 1409], [126, 652, 1408], [126, 656, 1409], [126, 660, 1411], [126, 663, 1413], [126, 666, 1414], [126, 670, 1417], [126, 674, 1419], [126, 677, 1421], [126, 680, 1424], [126, 683, 1427], [126, 686, 1429], [126, 689, 1431], [126, 692, 1433], [126, 694, 1436], [126, 698, 1437], [126, 701, 1441], [126, 704, 1443], [126, 708, 1446], [126, 711, 1449], [126, 714, 1451], [126, 717, 1453], [126, 720, 1455], [126, 722, 1458], [126, 726, 1461], [126, 730, 1463], [126, 733, 1466], [126, 736, 1469], [126, 739, 1471], [126, 743, 1473], [126, 746, 1476], [126, 750, 1477], 

In [14]:
sample_sc_seg_operator.save_seg_callback()

NameError: name 'sample_sc_seg_operator' is not defined

In [15]:
sample_sc_seg_operator.csn_correct_seg_callback()

NameError: name 'sample_sc_seg_operator' is not defined

current shape layer shape properties:  Event
setting face color of selected shape...
<selection complete>
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
setting face color of selected shape...
<selection complete>
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
current shap

c:\Users\Gaohan\anaconda3\envs\checkcuda\lib\site-packages\napari\layers\base\base.py:1632: RuntimeWarning: invalid value encountered in cast
  corners[:, displayed_axes] = data_bbox_clipped


[button] clear sc layer callback fired!
clear_sc_layer_callback done!


c:\Users\Gaohan\anaconda3\envs\checkcuda\lib\site-packages\napari\layers\base\base.py:1632: RuntimeWarning: invalid value encountered in cast
  corners[:, displayed_axes] = data_bbox_clipped


current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
current shape layer shape properties:  Event
shape already selected, please select another shape
toggle shapes text fired!
toggle shapes text fired!
toggle shapes text fired!
toggle shapes text fired!
toggle shapes text fired!
toggle shapes text fired!
restore sct shapes fired!
<restoring sct shapes>
clear selection callback fired!
clearing selection...
<clear complete>
clear selection callback fired!
clearing selection...
<clear complete>
switch mode callback fired!
mode changed to click&annotate
clearing selection...
<clear complete>
current shape layer shape properties:  Event
setting face color of selected shape...
<selection complete>
current shape layer shape properties:  Event
shape already selected, please select another shape
switch mode callback fired!
mode changed to connect
clearing 